# Analyse factorielle — Décrochage (G3=0)

Pipeline séquentiel en **3 boutons** :

1. **Réductions factorielles** → scree + tableau récap (60 % / 80 % / 90 %)
2. **Seuil de variance** + coude/silhouette + projection décrochage G3
3. **k-means** + caractérisation des clusters (3.1 à 3.4)

> **Note :** l'AFTD est exclue du notebook (trop d'axes MDS → graphiques peu lisibles). Disponible via CLI : `uv run python src/AFTD/main.py <run>`.

In [1]:
import sys
from pathlib import Path

SRC = Path("src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

sns.set_theme(style="whitegrid")

import config
from _utils import build_typology, detect_notebook_method_set, load_data
from config import (
    DATASET_LABELS, INCLUDE_VARS, SHARED_COLUMNS, SUBJECT_COLUMNS,
    apply_preprocessing, extract_dropout_mask, resolve_g3_column,
)
from factor_analysis.orchestrator import run_factor_analyses, build_variance_summary_table
from factor_analysis.inertia_viz import plot_inertia_comparison
from factor_analysis.dropout_viz import plot_pc1_pc2_dropout
from factor_analysis.clustering import run_clustering_pipeline
from CLUSTERING.kmeans import suggest_k_elbow, suggest_k_silhouette

%matplotlib inline

## Configuration (dataset + variables)

In [2]:
dataset_dd = widgets.Dropdown(
    options=[("Mathématiques", "student-mat"), ("Portugais", "student-por"), ("Global", "data_global")],
    value=config.DATASET,
    description="Dataset:",
)
avg_cb = widgets.Checkbox(value=config.AVERAGE_MAT_POR, description="Moyenner .m/.p (global)")
run_name_txt = widgets.Text(value="notebook_run", description="Run name:")
g3_dd = widgets.Dropdown(options=["G3"], description="Colonne G3:")

def update_g3_options(_=None):
    resolved = resolve_g3_column(dataset_dd.value, avg_cb.value)
    opts = [resolved] if isinstance(resolved, str) else list(resolved)
    g3_dd.options = opts
    g3_dd.value = opts[0]

dataset_dd.observe(update_g3_options, names="value")
avg_cb.observe(update_g3_options, names="value")
update_g3_options()

display(widgets.VBox([dataset_dd, avg_cb, run_name_txt, g3_dd]))

In [3]:
ALL_VARS = SHARED_COLUMNS + SUBJECT_COLUMNS
var_checks = {
    v: widgets.Checkbox(value=INCLUDE_VARS.get(v, True), description=v)
    for v in ALL_VARS
}
for v in ("G1", "G2", "G3"):
    var_checks[v].description = f"{v} (False recommandé)"

def preset_decrochage(_=None):
    for g in ("G1", "G2", "G3"):
        var_checks[g].value = False

def select_all(_=None):
    for cb in var_checks.values():
        cb.value = True

def select_none(_=None):
    for cb in var_checks.values():
        cb.value = False

def reset_config(_=None):
    for v, cb in var_checks.items():
        cb.value = INCLUDE_VARS.get(v, True)

btn_preset = widgets.Button(description="Preset décrochage")
btn_all = widgets.Button(description="Tout sélectionner")
btn_none = widgets.Button(description="Tout désélectionner")
btn_reset = widgets.Button(description="Réinitialiser config")
btn_preset.on_click(preset_decrochage)
btn_all.on_click(select_all)
btn_none.on_click(select_none)
btn_reset.on_click(reset_config)

methods_label = widgets.HTML("")
display(widgets.HBox([btn_preset, btn_all, btn_none, btn_reset]))
display(widgets.GridBox(
    list(var_checks.values()),
    layout=widgets.Layout(grid_template_columns="repeat(4, 1fr)"),
))
display(methods_label)

GridBox(children=(Checkbox(value=True, description='school'), Checkbox(value=True, description='sex'), Checkbo…

HTML(value='')

## Étape 1 — Réductions factorielles

Lancez les réductions, consultez le **graphique comparatif inertie** (par axe + cumulée, 20 axes max) et le **tableau récap** (colonnes 60 % / 80 % / 90 %) **avant** de choisir le seuil de variance à l'étape 2.

In [4]:
state = {}
method_dd = widgets.Dropdown(options=[], description="Méthode:")

def get_include_vars():
    return {v: cb.value for v, cb in var_checks.items()}

def update_methods_preview(_=None):
    from _utils import load_data  # noqa: PLC0415
    try:
        df = apply_preprocessing(
            load_data(dataset_dd.value),
            dataset=dataset_dd.value,
            include_vars=get_include_vars(),
            average_mat_por=avg_cb.value,
        )
        methods = detect_notebook_method_set(build_typology(df))
        methods_label.value = (
            f"<b>Méthodes (notebook) :</b> {', '.join(methods)}"
            "<br><i>AFTD exclue — voir CLI si besoin.</i>"
        )
    except Exception as e:
        methods_label.value = f"<span style='color:red'>{e}</span>"

for w in list(var_checks.values()) + [dataset_dd, avg_cb]:
    w.observe(update_methods_preview, names="value")
update_methods_preview()

out_factor = widgets.Output()
btn_factor = widgets.Button(
    description="1. Lancer les réductions factorielles", button_style="primary",
)

def run_factor_step(_=None):
    with out_factor:
        clear_output(wait=True)
        include = get_include_vars()
        g3_resolved = resolve_g3_column(dataset_dd.value, avg_cb.value)
        g3_col = g3_dd.value if isinstance(g3_resolved, list) else g3_resolved

        df_prep = apply_preprocessing(
            load_data(dataset_dd.value),
            dataset=dataset_dd.value,
            include_vars=include,
            average_mat_por=avg_cb.value,
        )
        methods = detect_notebook_method_set(build_typology(df_prep))

        results, typology, df_raw, df = run_factor_analyses(
            dataset=dataset_dd.value,
            include_vars=include,
            average_mat_por=avg_cb.value,
            run_name=run_name_txt.value,
            methods=methods,
            save=True,
        )
        state.clear()
        state.update({
            "results": results,
            "typology": typology,
            "df_raw": df_raw,
            "df": df,
            "g3_col": g3_col,
        })

        print(f"Dataset : {DATASET_LABELS[dataset_dd.value]} — {len(df_raw)} individus\n")
        for method, res in results.items():
            if res.contributions is not None and "contrib_dim1" in res.figures:
                display(Markdown(f"### {method} — contributions Dim1"))
                display(res.figures["contrib_dim1"])

        display(plot_inertia_comparison(
            results,
            title=f"Comparaison des méthodes — {DATASET_LABELS[dataset_dd.value]}",
        ))

        method_dd.options = list(results.keys())
        if method_dd.options:
            method_dd.value = method_dd.options[0]

        summary = build_variance_summary_table(results)
        state["summary"] = summary
        display(Markdown(
            "**Tableau récapitulatif** — pour ~80 % de variance, notez la colonne `80%` "
            "puis saisissez `0.80` à l'étape 2."
        ))
        display(summary.style.highlight_min(subset=["80%"], color="lightblue"))

btn_factor.on_click(run_factor_step)
display(widgets.VBox([btn_factor, out_factor]))

## Étape 2 — Seuil de variance et choix de k

### Coude et silhouette

Après le tableau récap, saisissez le **seuil de variance** (0–1, ex. `0.80`) et la **plage de k** à tester.

In [5]:
variance_threshold = widgets.FloatText(value=0.80, description="Seuil var.:")
k_min_txt = widgets.IntText(value=2, description="k min:")
k_max_txt = widgets.IntText(value=8, description="k max:")
out_step2 = widgets.Output()
btn_step2 = widgets.Button(
    description="2. Coude + silhouette + décrochage G3", button_style="info",
)

def run_step2(_=None):
    with out_step2:
        clear_output(wait=True)
        if "results" not in state:
            print("Lancez d'abord l'étape 1 (réductions factorielles).")
            return
        threshold = variance_threshold.value
        if not (0 < threshold <= 1):
            print("Seuil invalide : entre 0 et 1 (ex. 0.80).")
            return
        k_lo, k_hi = k_min_txt.value, k_max_txt.value
        if k_lo >= k_hi:
            print("k min doit être < k max.")
            return

        results = state["results"]
        df_raw = state["df_raw"]
        g3_col = state["g3_col"]

        print(f"Seuil variance = {threshold:.0%} → axes retenus par méthode :")
        for method, res in results.items():
            n_axes = res.select_coords(threshold=threshold).shape[1]
            print(f"  {method} : {n_axes} axes")

        from CLUSTERING.kmeans import compute_k_metrics, plot_elbow, plot_silhouette
        k_range = range(k_lo, k_hi + 1)
        metrics_by = {}
        for method, res in results.items():
            X = res.select_coords(threshold=threshold)
            metrics_by[method] = compute_k_metrics(X, k_range)
        state["metrics_by"] = metrics_by
        state["threshold"] = threshold
        state["k_lo"] = k_lo
        state["k_hi"] = k_hi

        display(Markdown("### Coude (inertie intra-classe)"))
        display(plot_elbow(metrics_by))
        display(Markdown("### Silhouette"))
        display(plot_silhouette(metrics_by))

        print("\nMétriques détaillées par méthode :")
        for method, m in metrics_by.items():
            display(Markdown(f"**{method}**"))
            display(m)

        print("\nSuggestions de k (coude / silhouette) :")
        for method, m in metrics_by.items():
            ke = suggest_k_elbow(m.set_index("k")["inertia"])
            ks = suggest_k_silhouette(m)
            print(f"  {method} — k coude={ke}, k silhouette={ks}")

        display(Markdown("### Projection décrochage (G3=0)"))
        dropout_mask = extract_dropout_mask(df_raw, g3_col)
        for method, res in results.items():
            display(plot_pc1_pc2_dropout(res.coords_df, dropout_mask, method, g3_col))

        print("\n→ Saisissez k final à l'étape 3.")

btn_step2.on_click(run_step2)
display(widgets.VBox([
    widgets.HTML("<i>Seuil : part de variance cumulée pour les axes avant clustering.</i>"),
    variance_threshold,
    widgets.HTML("<i>Plage de k pour coude et silhouette :</i>"),
    widgets.HBox([k_min_txt, k_max_txt]),
    btn_step2,
    out_step2,
]))

## Étape 3 — k-means et caractérisation des clusters

Après les courbes coude et silhouette, choisissez la **méthode** retenue et le **k final**.

- **3.1** Projection clusters PC1×PC2
- **3.2** Décrochage G3 par cluster
- **3.3** Heatmap z-score
- **3.4** Répartition de toutes les variables (grille compacte)

In [6]:
k_final_txt = widgets.IntText(value=4, description="k final:")
out_step3 = widgets.Output()
btn_step3 = widgets.Button(description="3. Lancer k-means + profils", button_style="success")

def run_step3(_=None):
    with out_step3:
        clear_output(wait=True)
        if "results" not in state or "metrics_by" not in state:
            print("Lancez d'abord les étapes 1 et 2.")
            return
        if not method_dd.options:
            print("Lancez d'abord l'étape 1 pour charger les méthodes.")
            return
        method = method_dd.value
        if method not in state["results"]:
            print(f"Méthode {method!r} invalide.")
            return
        k_final = k_final_txt.value
        k_lo = state.get("k_lo", k_min_txt.value)
        k_hi = state.get("k_hi", k_max_txt.value)
        if not (k_lo <= k_final <= k_hi):
            print(f"k final ({k_final}) doit être entre k min ({k_lo}) et k max ({k_hi}).")
            return

        k_range = range(k_lo, k_hi + 1)
        clust, _, _ = run_clustering_pipeline(
            state["results"], state["df"], state["df_raw"], state["typology"],
            k_range, state["threshold"], k_final, state["g3_col"],
            methods=[method],
        )
        state["clustering"] = clust
        cr = clust[method]

        display(Markdown(f"## {method} — k={cr.k}"))

        display(Markdown("#### 3.1 Projection clusters PC1×PC2"))
        if "cluster_projection" in cr.figures:
            display(cr.figures["cluster_projection"])

        display(Markdown("#### 3.2 Décrochage G3 par cluster"))
        if "dropout_by_cluster" in cr.profile_figures:
            display(cr.profile_figures["dropout_by_cluster"])

        display(Markdown("#### 3.3 Heatmap z-score"))
        if "zscore_heatmap" in cr.profile_figures:
            display(cr.profile_figures["zscore_heatmap"])

        display(Markdown("#### 3.4 Répartition de toutes les variables"))
        if "variable_repartitions_grid" in cr.profile_figures:
            display(cr.profile_figures["variable_repartitions_grid"])

btn_step3.on_click(run_step3)

display(widgets.VBox([
    widgets.HBox([method_dd, k_final_txt]),
    btn_step3,
    out_step3,
]))